# WORK9 — 04 Train Pair Models V0.2 — Corrected Hybrid + GPU + Rolling 3M Evaluation

**Scope:** candidate modeling only for `Base SKU × Branch × Month`.

V0.2 corrections from the audited V0.1 run:
- CatBoost Tweedie target scaling (`max(train target) → 1`) and prediction rescaling to M2.
- Behavior state uses **ADI + CV² + positive-history count**.
- Small behavior segments use the best allowed expert on the whole Primary horizon, not an arbitrary fixed fallback.
- Hurdle occurrence probability threshold is calibrated on VALIDATION PRIMARY by horizon.
- Add cumulative 3-month evaluation (Pair, Base SKU, Branch, Portfolio) using only complete H1/H2/H3 windows.
- Add rolling forecast-revision metrics (`H2→H1`, `H3→H2`) at Pair and Base-SKU levels.

Locked architecture remains:
- Baselines: Naive-1, SeasonalNaive-12, MovingAverage-3, Croston-SBA, TSB
- Global LightGBM Tweedie — direct H1/H2/H3
- Global CatBoost Tweedie — direct H1/H2/H3
- Hurdle + behavior-gated expert selection
- No deep learning

Safety:
- Supabase is not accessed by this notebook.
- TRAIN may include current-inactive historical entities.
- VALIDATION PRIMARY is current-active Base SKU × current-active Branch × known Pair.
- Frozen test / reconciliation / freeze / production are not touched.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Runtime dependencies. GPU is preferred but CPU fallback is built into the runner.
!pip -q install -U lightgbm catboost pyarrow pyyaml scikit-learn

import os, json, hashlib, sys, subprocess
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import yaml

WORK9 = Path('/content/drive/MyDrive/work9')
CONFIG = WORK9 / '01_config'
SRC = WORK9 / '02_src' / 'modeling'
REPORT_ROOT = WORK9 / '06_reports' / 'model_selection'
RUN_ROOT = WORK9 / '08_runs'

print('WORK9:', WORK9)
print('GPU check:')
os.system('nvidia-smi -L || true')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 109.3 MB/s eta 0:00:00
WORK9: /content/drive/MyDrive/work9
GPU check:


0

## 1. Verify lineage and SHA256 gates

This notebook uses only the current fresh Work9 Dataset/Feature/Feature-Selection pointers and does not hard-code an older run.


In [3]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

dataset_ptr = json.loads((CONFIG / 'current_dataset_run.json').read_text())
feature_ptr = json.loads((CONFIG / 'current_feature_run.json').read_text())
fs_ptr = json.loads((CONFIG / 'current_feature_selection_run.json').read_text())

assert dataset_ptr['status'] == 'PASS', dataset_ptr
assert dataset_ptr['dataset_version'] == 'dataset_v012'
assert feature_ptr['status'] == 'PASS', feature_ptr
assert fs_ptr['status'] == 'PASS', fs_ptr
assert feature_ptr['run_id'] == fs_ptr['source_feature_run_id'], (feature_ptr['run_id'], fs_ptr['source_feature_run_id'])
assert feature_ptr['pair_feature_version'] == 'pair_feature_v013'
assert fs_ptr['selection_version'] == 'feature_selection_v04'

PAIR_PANEL = Path(dataset_ptr['pair_panel_path'])
PAIR_FEATURE = Path(feature_ptr['pair_feature_panel_path'])
SELECTED = Path(fs_ptr['pair_selected_path'])
CONTRACT = CONFIG / 'model_contract_v02.yaml'

assert PAIR_PANEL.exists(), PAIR_PANEL
assert PAIR_FEATURE.exists(), PAIR_FEATURE
assert SELECTED.exists(), SELECTED
assert CONTRACT.exists(), CONTRACT

expected_pair_sha = feature_ptr['output_sha256']['pair_feature_panel']
actual_pair_sha = sha256_file(PAIR_FEATURE)
assert actual_pair_sha == expected_pair_sha, (actual_pair_sha, expected_pair_sha)

print('Current dataset run verified:', dataset_ptr['run_id'])
print('Current feature run verified:', feature_ptr['run_id'])
print('Current feature selection run verified:', fs_ptr['run_id'])
print('Pair feature SHA256 PASS:', actual_pair_sha)
print('Selected list:', SELECTED)


Current dataset run verified: core_dataset_v012_20260815T122509Z
Current feature run verified: feature_stage_v013_20260815T123431Z
Current feature selection run verified: feature_selection_v04_20260815T130048Z
Pair feature SHA256 PASS: 805d00b098b1d052516d95a469c4a83093d9425f89ea614683a71c726e780ace
Selected list: /content/drive/MyDrive/work9/05_selected_features/pair_selected_feature_list_v04.yaml


## 2. Static model-stage tests

Run these before spending GPU time.


In [4]:
!python -m pytest -q /content/drive/MyDrive/work9/07_tests/test_model_runner_v02.py


..................                                                       [100%]
18 passed in 3.36s


## 3. Train candidate models

The runner will try:
- LightGBM: `cuda → gpu → cpu`
- CatBoost: `GPU → CPU`

The actual device used is written into `model_manifest.json`.


In [5]:
sys.path.insert(0, str(SRC))
from model_runner_v02 import run_pair_model_stage

run_id = 'pair_modeling_v02_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report_dir = REPORT_ROOT / run_id
run_dir = RUN_ROOT / run_id
report_dir.mkdir(parents=True, exist_ok=False)
run_dir.mkdir(parents=True, exist_ok=False)

manifest = run_pair_model_stage(
    pair_feature_path=str(PAIR_FEATURE),
    pair_panel_path=str(PAIR_PANEL),
    selected_feature_path=str(SELECTED),
    contract_path=str(CONTRACT),
    output_dir=str(report_dir),
    run_id=run_id,
    save_models=True,
)

print(json.dumps(manifest['row_counts'], indent=2))
print('Primary behavior-gated:', json.dumps(manifest['primary_behavior_gated'], indent=2))
print('Devices:')
for d in manifest['devices']:
    print(d)


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


0:	learn: 1.8127807	test: 1.8199700	best: 1.8199700 (0)	total: 81.9ms	remaining: 1m 54s
100:	learn: 0.1701590	test: 0.1886249	best: 0.1886249 (100)	total: 6.74s	remaining: 1m 26s
200:	learn: 0.1620104	test: 0.1854658	best: 0.1854658 (200)	total: 12.3s	remaining: 1m 13s
300:	learn: 0.1591738	test: 0.1834238	best: 0.1834238 (300)	total: 17.4s	remaining: 1m 3s
400:	learn: 0.1574595	test: 0.1822971	best: 0.1822971 (400)	total: 26.1s	remaining: 1m 5s
500:	learn: 0.1555144	test: 0.1811480	best: 0.1811390 (499)	total: 31.3s	remaining: 56.1s
600:	learn: 0.1541674	test: 0.1807656	best: 0.1806513 (583)	total: 37.8s	remaining: 50.2s
700:	learn: 0.1531971	test: 0.1800914	best: 0.1800412 (694)	total: 43s	remaining: 42.9s
800:	learn: 0.1522750	test: 0.1794766	best: 0.1793949 (777)	total: 49.2s	remaining: 36.8s
900:	learn: 0.1513851	test: 0.1788890	best: 0.1788890 (900)	total: 54.8s	remaining: 30.4s
1000:	learn: 0.1506323	test: 0.1783208	best: 0.1783197 (999)	total: 60s	remaining: 23.9s
1100:	learn: 

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

0:	learn: 1.8228466	test: 1.8259591	best: 1.8259591 (0)	total: 93.5ms	remaining: 2m 10s
100:	learn: 0.1914338	test: 0.1553536	best: 0.1552978 (99)	total: 5.25s	remaining: 1m 7s
200:	learn: 0.1827153	test: 0.1362574	best: 0.1362574 (200)	total: 11.9s	remaining: 1m 10s
300:	learn: 0.1790428	test: 0.1317284	best: 0.1316836 (298)	total: 17s	remaining: 1m 2s
400:	learn: 0.1751007	test: 0.1267832	best: 0.1267832 (400)	total: 22.3s	remaining: 55.5s
500:	learn: 0.1732568	test: 0.1244329	best: 0.1244085 (490)	total: 28.7s	remaining: 51.4s
600:	learn: 0.1717552	test: 0.1218648	best: 0.1218637 (599)	total: 33.8s	remaining: 44.9s
700:	learn: 0.1709333	test: 0.1206284	best: 0.1206284 (700)	total: 40.7s	remaining: 40.6s
800:	learn: 0.1701732	test: 0.1196871	best: 0.1196274 (797)	total: 45.9s	remaining: 34.3s
900:	learn: 0.1695890	test: 0.1187121	best: 0.1187121 (900)	total: 51.7s	remaining: 28.6s
1000:	learn: 0.1689044	test: 0.1179532	best: 0.1179214 (968)	total: 57.9s	remaining: 23.1s
1100:	learn: 

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/loc

0:	learn: 1.8193843	test: 1.8202905	best: 1.8202905 (0)	total: 91.8ms	remaining: 2m 8s
100:	learn: 0.1848945	test: 0.2178384	best: 0.2178384 (100)	total: 5.47s	remaining: 1m 10s
200:	learn: 0.1749574	test: 0.2082425	best: 0.2082425 (200)	total: 11.5s	remaining: 1m 8s
300:	learn: 0.1707776	test: 0.2040518	best: 0.2040518 (300)	total: 16.5s	remaining: 1m
400:	learn: 0.1684061	test: 0.2014536	best: 0.2014078 (399)	total: 23s	remaining: 57.4s
500:	learn: 0.1667584	test: 0.1991127	best: 0.1990625 (493)	total: 28s	remaining: 50.3s
600:	learn: 0.1654288	test: 0.1972318	best: 0.1972206 (597)	total: 33.3s	remaining: 44.3s
700:	learn: 0.1646316	test: 0.1960746	best: 0.1960746 (700)	total: 39.4s	remaining: 39.3s
800:	learn: 0.1638213	test: 0.1950351	best: 0.1950091 (789)	total: 44.5s	remaining: 33.3s
900:	learn: 0.1631976	test: 0.1940562	best: 0.1940513 (899)	total: 51s	remaining: 28.3s
1000:	learn: 0.1626045	test: 0.1931957	best: 0.1931791 (999)	total: 56.1s	remaining: 22.4s
1100:	learn: 0.16183

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{
  "train": 1090317,
  "validation_primary": 44289,
  "validation_secondary": 77886
}
Primary behavior-gated: {
  "model": "behavior_gated",
  "universe": "PRIMARY",
  "horizon": null,
  "segment": null,
  "n_rows": 44289,
  "wape": 0.8593796511853881,
  "mae": 25.086772924307947,
  "bias": -825318.3270370018,
  "bias_ratio": -0.6383603173509279,
  "zero_false_positive_rate": 0.030004110152075627,
  "positive_wape": 0.8143224891452846
}
Devices:
{'library': 'lightgbm', 'requested': 'cuda', 'actual': 'gpu', 'fallback_reason': 'cuda:LightGBMError:CUDA Tree Learner was not enabled in this build.\nPlease recompile with CMake option -DUSE_CUDA=1 (NVIDIA GPUs) or -DUSE_ROCM=1 (AMD GPUs)', 'horizon': 1, 'candidate': 'lightgbm_tweedie'}
{'library': 'catboost', 'requested': 'GPU', 'actual': 'GPU', 'fallback_reason': None, 'horizon': 1, 'candidate': 'catboost_tweedie'}
{'library': 'lightgbm', 'requested': 'cuda', 'actual': 'gpu', 'fallback_reason': 'cuda:LightGBMError:CUDA Tree Learner was not 

## 4. Audit scoreboard before accepting pointer

Primary ranking is the only model-selection universe. Secondary-all remains diagnostic only.

Forecast revision is **optional/evaluable only when the same target month is forecast from consecutive origins**. An empty revision table is a valid `NOT EVALUABLE` condition and must not fail the run.


In [6]:
# Recover the latest completed V0.2 report if this notebook was reopened after training.
if 'report_dir' not in globals() or not Path(report_dir).exists():
    completed = sorted(
        [p for p in REPORT_ROOT.glob('pair_modeling_v02_*') if (p / 'model_manifest.json').exists()]
    )
    if not completed:
        raise FileNotFoundError('No completed pair_modeling_v02 report found.')
    report_dir = completed[-1]
    run_id = report_dir.name
    run_dir = RUN_ROOT / run_id
    manifest = json.loads((report_dir / 'model_manifest.json').read_text(encoding='utf-8'))

score = pd.read_csv(report_dir / 'model_scoreboard_primary.csv')
display(score.sort_values(['horizon','segment','wape'], na_position='last').head(60))

gate = pd.read_csv(report_dir / 'behavior_gate.csv')
display(gate)

hurdle_cal = pd.read_csv(report_dir / 'hurdle_threshold_calibration.csv')
display(hurdle_cal.sort_values(['horizon','wape']).head(30))

cum3 = pd.read_csv(report_dir / 'cumulative_3m_scoreboard.csv')
display(cum3.sort_values(['level','wape_3m']).head(50))

revision_path = report_dir / 'forecast_revision_scoreboard.csv'
try:
    revision = pd.read_csv(revision_path)
except pd.errors.EmptyDataError:
    # Backward-compatible recovery for the first V0.2 run, which wrote a 1-byte
    # CSV when no consecutive-origin revision pairs existed.
    revision = pd.DataFrame(columns=[
        'model','level','transition','n_units','revision_mae_m2',
        'revision_ratio_vs_old_forecast','signed_revision_m2'
    ])

if revision.empty:
    print('Forecast revision: NOT EVALUABLE on this Primary Validation window '
          '(no same target month forecast from consecutive origins).')
else:
    display(revision.sort_values(
        ['level','transition','revision_ratio_vs_old_forecast']
    ).head(50))

print('3M coverage:', json.dumps(manifest['cumulative_3m']['coverage'], indent=2))
print('Revision status:', json.dumps(manifest.get('forecast_revision', {}), indent=2))


,model,universe,horizon,segment,n_rows,wape,mae,bias,bias_ratio,zero_false_positive_rate,positive_wape
70,hurdle,PRIMARY,1.0,NaN,14775,0.793740,30.796961,-2.835170e+05,-0.494563,0.072721,0.735821
77,behavior_gated,PRIMARY,1.0,NaN,14775,0.793740,30.796961,-2.835170e+05,-0.494563,0.072721,0.735821
56,lightgbm_tweedie,PRIMARY,1.0,NaN,14775,0.876758,34.018048,-1.930566e+05,-0.336765,1.000000,0.704479
63,catboost_tweedie,PRIMARY,1.0,NaN,14775,0.908069,35.232930,-2.063818e+05,-0.360010,1.000000,0.718258
9,moving_average_3,PRIMARY,1.0,NaN,14775,0.915929,35.537880,-5.720995e+04,-0.099796,0.353675,0.725614
35,moving_average_3,PRIMARY,1.0,NaN,14775,0.915929,35.537880,-5.720995e+04,-0.099796,0.353675,0.725614
1,naive_1,PRIMARY,1.0,NaN,14775,1.066594,41.383664,5.486386e+04,0.095704,0.179872,0.881640
21,naive_1,PRIMARY,1.0,NaN,14775,1.066594,41.383664,5.486386e+04,0.095704,0.179872,0.881640
5,seasonal_naive_12,PRIMARY,1.0,NaN,14775,1.224119,47.495600,-3.027714e+05,-0.528150,0.202604,0.983442
28,seasonal_naive_12,PRIMARY,1.0,NaN,14775,1.224119,47.495600,-3.027714e+05,-0.528150,0.202604,0.983442


,horizon,behavior_segment,n_rows,selection_scope,chosen_expert,selection_wape,segment_wape
0,1,regular,1369,SEGMENT_PRIMARY,hurdle,0.640795,0.640795
1,1,intermittent,11346,SEGMENT_PRIMARY,hurdle,0.860904,0.860904
2,1,very_sparse,2060,SEGMENT_PRIMARY,hurdle,1.020349,1.020349
3,2,regular,1276,SEGMENT_PRIMARY,hurdle,0.889521,0.889521
4,2,intermittent,11153,SEGMENT_PRIMARY,hurdle,0.982385,0.982385
5,2,very_sparse,2055,SEGMENT_PRIMARY,hurdle,1.000000,1.000000
6,3,regular,1404,SEGMENT_PRIMARY,hurdle,0.754490,0.754490
7,3,intermittent,11552,SEGMENT_PRIMARY,hurdle,0.970704,0.970704
8,3,very_sparse,2074,SEGMENT_PRIMARY,hurdle,1.001276,1.001276


,threshold,wape,zero_false_positive_rate,mae,bias_ratio,horizon
10,0.50,0.793740,0.072721,30.796961,-0.494563,1
9,0.45,0.794281,0.094460,30.817957,-0.462255,1
11,0.55,0.794338,0.051755,30.820185,-0.530581,1
12,0.60,0.798236,0.036747,30.971433,-0.564778,1
8,0.40,0.798918,0.122710,30.997868,-0.431847,1
7,0.35,0.801738,0.154160,31.107294,-0.408440,1
13,0.65,0.802946,0.025160,31.154171,-0.607288,1
6,0.30,0.806346,0.189362,31.286109,-0.390207,1
14,0.70,0.813497,0.016884,31.563549,-0.647018,1
5,0.25,0.814283,0.230633,31.594032,-0.371698,1


,model,level,n_units,wape_3m,mae_3m,bias_3m,bias_ratio_3m,actual_sum_m2,forecast_sum_m2
14,lightgbm_tweedie,BASE_SKU,1227,0.415513,4.036841e+02,-3.045602e+05,-0.255488,1192070.63,8.875105e+05
11,moving_average_3,BASE_SKU,1227,0.438279,4.258026e+02,2.278074e+05,0.191102,1192070.63,1.419878e+06
15,catboost_tweedie,BASE_SKU,1227,0.468300,4.549687e+02,4.736501e+03,0.003973,1192070.63,1.196807e+06
9,naive_1,BASE_SKU,1227,0.638871,6.206841e+02,5.186872e+05,0.435114,1192070.63,1.710758e+06
16,hurdle,BASE_SKU,1227,0.656259,6.375773e+02,-7.656553e+05,-0.642290,1192070.63,4.264153e+05
17,behavior_gated,BASE_SKU,1227,0.656259,6.375773e+02,-7.656553e+05,-0.642290,1192070.63,4.264153e+05
10,seasonal_naive_12,BASE_SKU,1227,0.972315,9.446356e+02,-4.042334e+05,-0.339102,1192070.63,7.878372e+05
13,tsb,BASE_SKU,1227,1.491556,1.449096e+03,1.735860e+06,1.456172,1192070.63,2.927930e+06
12,croston_sba,BASE_SKU,1227,2.270238,2.205611e+03,2.669411e+06,2.239306,1192070.63,3.861481e+06
24,catboost_tweedie,BRANCH,57,0.259760,5.432493e+03,4.736501e+03,0.003973,1192070.63,1.196807e+06


Forecast revision: NOT EVALUABLE on this Primary Validation window (no same target month forecast from consecutive origins).
3M coverage: {
  "pair_origin_total": 15076,
  "pair_origin_complete_h1_h2_h3": 14166,
  "pair_origin_incomplete": 910,
  "complete_rate": 0.9396391615813213
}
Revision status: {
  "transitions": [
    "H2_TO_H1",
    "H3_TO_H2"
  ],
  "evaluable": false,
  "not_evaluable_reason": "NO_CONSECUTIVE_ORIGIN_TARGET_OVERLAP_IN_PRIMARY_VALIDATION",
  "rows": []
}


## 5. Write run manifest + current candidate pointer

This pointer means **candidate model run exists**. It does not mean frozen test, freeze, reconciliation or production are authorized.


In [7]:
# This cell can be rerun after the audit cell without retraining.
if 'report_dir' not in globals() or not Path(report_dir).exists():
    completed = sorted(
        [p for p in REPORT_ROOT.glob('pair_modeling_v02_*') if (p / 'model_manifest.json').exists()]
    )
    if not completed:
        raise FileNotFoundError('No completed pair_modeling_v02 report found.')
    report_dir = completed[-1]
    run_id = report_dir.name
    run_dir = RUN_ROOT / run_id

manifest_path = report_dir / 'model_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['status'] == 'PASS', manifest
assert manifest['model_version'] == 'pair_modeling_v02', manifest
manifest_sha = sha256_file(manifest_path)

dataset_ptr = json.loads((CONFIG / 'current_dataset_run.json').read_text())
feature_ptr = json.loads((CONFIG / 'current_feature_run.json').read_text())
fs_ptr = json.loads((CONFIG / 'current_feature_selection_run.json').read_text())

run_manifest = {
    'run_id': run_id,
    'run_type': 'PAIR_MODEL_CANDIDATE_V02',
    'status': 'PASS',
    'source_feature_run_id': feature_ptr['run_id'],
    'source_feature_selection_run_id': fs_ptr['run_id'],
    'report_dir': str(report_dir),
    'model_manifest_path': str(manifest_path),
    'model_manifest_sha256': manifest_sha,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'safety': {
        'supabase_accessed': False,
        'frozen_test_touched': False,
        'reconciliation_run': False,
        'production_published': False,
    },
}
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

pointer = {
    'run_id': run_id,
    'status': 'PASS',
    'model_version': 'pair_modeling_v02',
    'source_feature_run_id': feature_ptr['run_id'],
    'source_feature_selection_run_id': fs_ptr['run_id'],
    'report_dir': str(report_dir),
    'run_manifest_path': str(run_dir / 'run_manifest.json'),
    'model_manifest_path': str(manifest_path),
    'model_manifest_sha256': manifest_sha,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(CONFIG / 'current_model_candidate_run.json').write_text(json.dumps(pointer, indent=2), encoding='utf-8')

print('MODEL STAGE PASS:', run_id)
print('Pointer:', CONFIG / 'current_model_candidate_run.json')
print('STOP HERE for model audit. Do not touch frozen test yet.')


MODEL STAGE PASS: pair_modeling_v02_20260815T130724Z
Pointer: /content/drive/MyDrive/work9/01_config/current_model_candidate_run.json
STOP HERE for model audit. Do not touch frozen test yet.
